<a href="https://colab.research.google.com/github/maggie-changg/dsci550-assignment1/blob/main/Task_5a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
import pandas as pd
# Load datasets
ucr_by_state_df = pd.read_csv("../data/ucr_by_state.csv")
# Read the CSV file without headers (treats the first row as data)
haunted_places_df = pd.read_csv("../data/haunted_places.tsv", sep=",", header=None)

# Split the first row into column names
haunted_places_df.columns = haunted_places_df.iloc[0]  # Use first row as headers
haunted_places_df = haunted_places_df[1:]  # Remove the first row from data

# Reset index
haunted_places_df.reset_index(drop=True, inplace=True)

In [91]:
# 1. Data Cleaning and Preparation
# Convert relevant columns to numeric (removing commas)
ucr_by_state_df["violent_crime_total"] = ucr_by_state_df["violent_crime_total"].astype(str).str.replace(",", "").astype(float)
ucr_by_state_df["murder_manslaughter"] = ucr_by_state_df["murder_manslaughter"].astype(str).str.replace(",", "").astype(float)
ucr_by_state_df["property_crime_total"] = ucr_by_state_df["property_crime_total"].astype(str).str.replace(",", "").astype(float)

In [92]:
# 2. Compute the average crime statistics for each state across all years
# Compute the average crime statistics per state
ucr_avg_df = ucr_by_state_df.groupby("jurisdiction").agg({
    "violent_crime_total": "mean",
    "murder_manslaughter": "mean",
    "property_crime_total": "mean"
}).reset_index()

# Rename columns for clarity
ucr_avg_df.rename(columns={
    "jurisdiction": "State",
    "violent_crime_total": "avg_violent_crime",
    "murder_manslaughter": "avg_murder_manslaughter",
    "property_crime_total": "avg_property_crime"
}, inplace=True)

In [93]:
# Rename State column properly
for col in haunted_places_df.columns:
    if "state" in col.lower():
        haunted_places_df.rename(columns={col: "State"}, inplace=True)
        break

Now, both datasets have a “State” column, so we can merge them.

In [94]:
# Merge Haunted Places dataset with UCR crime data on 'State'
merged_df = haunted_places_df.merge(ucr_avg_df, on="State", how="left")

In [95]:
# Save the merged dataset as a new file
merged_df.to_csv("merged_haunted_places_1.csv", index=False, header=True)
# Display first few rows
print(merged_df.head())

      city        country                                        description  \
0      Ada  United States  Ada witch - Sometimes you can see a misty blue...   
1  Addison  United States  A little girl was killed suddenly while waitin...   
2   Adrian  United States  If you take Gorman Rd. west towards Sand Creek...   
3   Adrian  United States  In the 1970's, one room, room 211, in the old ...   
4   Albion  United States  Kappa Delta Sorority - The Kappa Delta Sororit...   

                   location     State state_abbrev           longitude  \
0              Ada Cemetery  Michigan           MI  -85.50489309999999   
1           North Adams Rd.  Michigan           MI         -84.3818434   
2             Ghost Trestle  Michigan           MI  -84.03565619999999   
3  Siena Heights University  Michigan           MI         -84.0175653   
4            Albion College  Michigan           MI         -84.7451775   

     latitude      city_longitude city_latitude  ...  \
0  42.9621061     